# local-rag 라벨링 도구 (Google Colab)

Colab GPU에서 추출(OCR)·라벨링 서버를 띄우고 브라우저로 접속합니다.

**런타임 → 런타임 유형 변경 → GPU** 를 켜 두세요.

### `reinstall_venv` 와의 관계

`scripts/reinstall_venv.bat` 는 **Windows PC끼리** `.venv` 경로/인터프리터가 깨졌을 때
`Lib`(site-packages)만 살리고 Scripts를 다시 묶는 용도입니다.

Colab은 Linux + 세션마다 디스크가 초기화되므로 그 스크립트를 그대로 쓰지 않습니다.
여기에서는 `pip install` 로 의존성을 깔고, **Drive에 라벨링 결과(`data/labeling`)만 백업**합니다.

## 1. Drive 마운트 + 프로젝트 경로

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Drive에 올려 둔 프로젝트 루트 (필요하면 경로만 수정)
# 예: /content/drive/MyDrive/local-rag
DRIVE_ROOT = Path("/content/drive/MyDrive/local-rag")
WORK = Path("/content/local-rag")

assert DRIVE_ROOT.is_dir(), f"Drive에 프로젝트가 없습니다: {DRIVE_ROOT}"

# 작업은 Colab 로컬 디스크에서 (SQLite 잠금·속도)
import os, shutil
if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(
    DRIVE_ROOT,
    WORK,
    ignore=shutil.ignore_patterns(
        ".venv", "__pycache__", ".git", "models", "llama", "*.gguf",
        "node_modules",
    ),
)
os.chdir(WORK)
print("작업 디렉터리:", WORK)

## 2. 의존성 설치 (메인 + 라벨러)

Windows `.venv/Lib` 를 가져오지 마세요 — OS/아키텍처가 달라 깨집니다.

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# OpenDataLoader PDF 용 Java
!apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless

pip("-r", "requirements.txt")
pip("-r", "tools/labeler/requirements.txt")

import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Drive에서 라벨링 데이터 복원 (있으면)

In [ ]:
from pathlib import Path
import shutil

src = DRIVE_ROOT / "data" / "labeling"
dst = WORK / "data" / "labeling"
dst.mkdir(parents=True, exist_ok=True)

if (src / "labeler.sqlite3").is_file():
    shutil.copy2(src / "labeler.sqlite3", dst / "labeler.sqlite3")
    print("DB 복원됨")
else:
    print("Drive에 기존 DB 없음 — 새로 시작")

src_img = src / "images"
if src_img.is_dir():
    if (dst / "images").exists():
        shutil.rmtree(dst / "images")
    shutil.copytree(src_img, dst / "images")
    print("images 복원됨")

# PDF는 Drive 원본을 그대로 써도 되고, 작업 복사본 data/docs 를 씀
docs = list((WORK / "data" / "docs").rglob("*.pdf")) if (WORK / "data" / "docs").exists() else []
print(f"PDF {len(docs)}개")

## 4. 라벨러 서버 기동 + Colab 프록시 URL

셀을 실행한 뒤 출력된 URL을 클릭하세요. (같은 Colab 세션 브라우저에서만 동작)

In [ ]:
import threading, time, subprocess, sys
from pathlib import Path
from google.colab.output import eval_js

PORT = 8788
server_proc = None

def run_server():
    global server_proc
    server_proc = subprocess.Popen(
        [sys.executable, "tools/labeler/server.py"],
        cwd=str(WORK),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in server_proc.stdout:
        print(line, end="")

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(3)

url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
print("\n라벨링 UI:", url)

## 5. Drive로 라벨링 결과 백업 (작업 중간·종료 전 실행)

세션이 끊기면 Colab 로컬 디스크는 사라집니다. 자주 실행하세요.

In [ ]:
from pathlib import Path
import shutil
from datetime import datetime, timezone

src = WORK / "data" / "labeling"
dst = DRIVE_ROOT / "data" / "labeling"
dst.mkdir(parents=True, exist_ok=True)

db = src / "labeler.sqlite3"
if db.is_file():
    shutil.copy2(db, dst / "labeler.sqlite3")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    shutil.copy2(db, dst / f"labeler.{stamp}.sqlite3.bak")

img_src = src / "images"
img_dst = dst / "images"
if img_src.is_dir():
    if img_dst.exists():
        shutil.rmtree(img_dst)
    shutil.copytree(img_src, img_dst)

print("백업 완료 →", dst)

## (선택) VLM 설명 — Colab에서 GPU로

로컬 VL llama-server 대신, transformers로 Qwen2.5-VL을 직접 띄우는 방식도 가능합니다.
메모리·설치 시간이 크므로 필요할 때만 사용하세요.

지금은 라벨 UI의 [VLM 설명 생성]이 `VLM_BASE_URL`(기본 `localhost:8090`)을 기대합니다.
Colab에서 VL을 쓰려면 별도 셀로 VL API를 띄운 뒤:

```python
import os
os.environ["VLM_BASE_URL"] = "http://127.0.0.1:8090/v1"
```

환경으로 서버를 재시작한 뒤 버튼을 누르면 됩니다. (VL 서버 셀은 추후 확장)